In [2]:
import numpy as np
import pandas as pd
import statsmodels
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint, adfuller
import matplotlib.pyplot as plt
import seaborn as sns; sns.set(style="whitegrid")
import datetime
import yfinance as yf

In [3]:
start = datetime.datetime(2010, 1, 1)
end = datetime.datetime.today().strftime('%Y-%m-%d')

tickers = ['AAPL', 'ADBE', 'ORCL', 'EBAY', 'MSFT', 'QCOM', 'HPQ', 'AMD', 'IBM','NVDA','INTC']


df = yf.download(tickers, start, end)['Close']
df.tail()

[*********************100%***********************]  11 of 11 completed


Ticker,AAPL,ADBE,AMD,EBAY,HPQ,IBM,INTC,MSFT,NVDA,ORCL,QCOM
Date,,,,,,,,,,,
2026-09-23,337.019989,240.690002,614.609985,109.089996,32.160000,232.759995,122.599998,500.589996,225.509995,144.559998,197.240005
2026-09-24,335.920013,238.929993,629.260010,107.830002,31.139999,227.059998,127.389999,497.929993,224.580002,139.539993,194.259995
2026-09-25,341.070007,235.470001,630.630005,107.900002,31.299999,225.509995,123.000000,516.169983,225.070007,137.100006,201.970001
2026-09-28,338.399994,231.009995,607.869995,107.320000,31.320000,220.669998,116.029999,509.220001,228.860001,132.600006,187.479996
2026-09-29,329.399994,233.169998,607.570007,105.209999,30.730000,219.990005,115.930000,508.959991,227.210007,137.789993,184.100006


Now we test our stocks for cointegraion using the pvalue and store the pairs for future use

In [4]:
def find_pairs(df):
    n=df.shape[1]
    ticker=df.keys()
    pairs=[]
    p_values={}
    for i in range(n):
        for j in range(i+1,n):
            p_value=coint(df[ticker[i]],df[ticker[j]])[1]
            if p_value<0.05:
                pairs.append((ticker[i],ticker[j]))
            p_values[(ticker[i],ticker[j])]=p_value
    return pairs,p_values

In [5]:
train_test_split=0.7
split=int(len(df)*train_test_split)
pairs,p_values_train=find_pairs(df[:split])

In [6]:
print(pairs)
print(p_values_train)

[('ADBE', 'MSFT'), ('EBAY', 'ORCL')]
{('AAPL', 'ADBE'): 0.2837422604473285, ('AAPL', 'AMD'): 0.18282880177310773, ('AAPL', 'EBAY'): 0.19393430621597274, ('AAPL', 'HPQ'): 0.6903129008219987, ('AAPL', 'IBM'): 1.0, ('AAPL', 'INTC'): 0.8935796523804879, ('AAPL', 'MSFT'): 0.09243586757668998, ('AAPL', 'NVDA'): 0.37742721810845375, ('AAPL', 'ORCL'): 0.10959717249463657, ('AAPL', 'QCOM'): 0.34924050824072317, ('ADBE', 'AMD'): 0.3482783731187764, ('ADBE', 'EBAY'): 0.2667192496613931, ('ADBE', 'HPQ'): 0.7152790442752021, ('ADBE', 'IBM'): 1.0, ('ADBE', 'INTC'): 0.940017409985906, ('ADBE', 'MSFT'): 0.01548115031278116, ('ADBE', 'NVDA'): 0.8563714883020241, ('ADBE', 'ORCL'): 0.059938534049934926, ('ADBE', 'QCOM'): 0.6592856992428909, ('AMD', 'EBAY'): 0.41222001516448803, ('AMD', 'HPQ'): 0.7459109569836752, ('AMD', 'IBM'): 1.0, ('AMD', 'INTC'): 0.9561335083261715, ('AMD', 'MSFT'): 0.08679619756769918, ('AMD', 'NVDA'): 0.40344631163588196, ('AMD', 'ORCL'): 0.26361976880431537, ('AMD', 'QCOM'): 0.224

In [1]:
def trade_ratios(df,pairs,train_test_split=0.7,window_long=60,window_short=5):
    if train_test_split>1 or train_test_split<0 or window_long==0 or window_short==0:
        return 0
    split=int(len(df)*train_test_split)
    results={}
    for i in range (len(pairs)):
        S1=df[pairs[i][0]].iloc[split:]
        S2=df[pairs[i][1]].iloc[split:]
        ratio=S1/S2
        ma_long=ratio.rolling(window=window_long).mean()
        ma_short=ratio.rolling(window=window_short).mean()
        std=ratio.rolling(window=window_long).std()
        zscore=(ma_short-ma_long)/std

        money=0
        countS1=0
        countS2=0

        for j in range(len(ratio)):
            if zscore.iloc[j]>1:
                money += S1.iloc[j] - S2.iloc[j] * ratio.iloc[j]
                countS1 -= 1
                countS2 += ratio.iloc[j]
            elif zscore.iloc[j] < -1:
                money -= S1.iloc[j] - S2.iloc[j] * ratio.iloc[j]
                countS1 += 1
                countS2 -= ratio.iloc[j]
            elif abs(zscore.iloc[j]) < 0.75:
                money += S1.iloc[j] * countS1 + S2.iloc[j] * countS2
                countS1 = 0
                countS2 = 0

        results[(pairs[i][0],pairs[i][1])]=money
        print(f"{pairs[i]}'s return : {money}")
    return results

In [7]:
results=trade_ratios(df,pairs)

('ADBE', 'MSFT')'s return : -2043.640383298548
('EBAY', 'ORCL')'s return : 1392.5788874370546


In [8]:
def test_coint(pairs,df):
    p_values={}
    for i in range (len(pairs)):
        p_value=coint(df[pairs[i][0]],df[pairs[i][1]])[1]
        if p_value>0.05:
            print(f"{pairs[i]} is not cointegrated on test data,p_value: {p_value}")
        else :
            print(f"{pairs[i]} is cointegrated on test data, p_value: {p_value}")
        p_values[(pairs[i][0],pairs[i][1])]=p_value
    return p_values

In [9]:
p_values_test=test_coint(pairs,df.iloc[split:])

('ADBE', 'MSFT') is not cointegrated on test data,p_value: 0.5783430439511195
('EBAY', 'ORCL') is not cointegrated on test data,p_value: 0.9413744260863244


We see that our pairs were not cointegrated but the pair EBAY and ORCL still made 1.4k on the test data.